In [1]:
%%capture
!pip install langchain langchain-community langchain-huggingface langchain-qdrant qdrant-client sentence-transformers langchain-text-splitters google-genai pydantic openai
!pip install -U transformers accelerate bitsandbytes
!pip install ragas==0.1.21 langchain-openai xmltodict

In [16]:
import os
import glob
import pickle
from google.colab import userdata

from langchain_core.documents import Document
from langchain_text_splitters import MarkdownTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http import models
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import torch

from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional, List
import re

from langchain_openai import ChatOpenAI
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from ragas.run_config import RunConfig

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
QDRANT_URL = userdata.get('QDRANT_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
# GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
HF_TOKEN = userdata.get('HF_TOKEN')
EMBEDDING_MODEL_ID = "google/embeddinggemma-300m"
LLM_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
COLLECTION_NAME = "medical_reports_v1"
DATA_FOLDER_PATH = '/content/drive/Othercomputers/My Laptop/knowledge-base/processed-pymupdf'
PICKLE_BACKUP_PATH = "backup_chunks_medis_v1.pkl"
CHAR_LENGTH = 5000

In [5]:
def extract_section(md_content, start_markers, stop_markers, length=None):
    if not md_content:
        return ""

    # Normalisasi input
    if isinstance(start_markers, str):
        start_markers = [start_markers]
    if isinstance(stop_markers, str):
        stop_markers = [stop_markers]

    # Gabungkan marker
    start_combined = "|".join(re.escape(m) for m in start_markers)
    stop_combined = "|".join(re.escape(m) for m in stop_markers)

    # Regex start:
    # - tidak wajib awal baris
    # - toleran bold/header
    start_pattern = re.compile(
        rf"(?i)(?:^|\n|\|)\s*[_*#\s]*\b(?:{start_combined})\b[_*\s]*(?:\([^)]*\))?",
        re.MULTILINE
    )

    matches = list(start_pattern.finditer(md_content))
    if not matches:
        return ""

    start_match = matches[-1]

    # start_match = start_pattern.search(md_content)
    if not start_match:
        return ""

    # Mulai setelah judul start
    content_start = start_match.end()

    # Regex stop (judul berikutnya)
    stop_pattern = re.compile(
        rf"(?i)(?:^|\n|\|)\s*[_*#\s]*\b(?:{stop_combined})\b[_*\s]*(?:\([^)]*\))?",
        re.MULTILINE
    )

    stop_match = stop_pattern.search(md_content, content_start)

    result = md_content[content_start:stop_match.start()] if stop_match else md_content[content_start:]
    result = result.strip()
    return result[:length] if length else result


In [6]:
client = OpenAI(api_key=OPENAI_API_KEY)

In [7]:
class Antropometri(BaseModel):
    jenis_kelamin: str
    usia: str
    berat_badan: str
    tinggi_badan: str
    lingkar_kepala: Optional[str] = None
    lingkar_lengan_atas: Optional[str] = None
    indeks_massa_tubuh: Optional[str] = None

class DataMedis(BaseModel):
    anthropometric_data: Antropometri = Field(..., description="Objek data antropometri")
    medical_diagnosis: str = Field(..., description="Diagnosis medis")

    @property
    def anthropometric_string(self) -> str:
        # Mengubah object menjadi dictionary, hapus yang None, lalu gabung jadi string
        data_dict = self.anthropometric_data.model_dump(exclude_none=True)
        # Format: Key: Value, Key: Value
        formatted_list = [f"{k.replace('_', ' ').title()}: {v}" for k, v in data_dict.items()]
        return ", ".join(formatted_list)

In [8]:
print(f"Sedang membaca file dari: {DATA_FOLDER_PATH}/*.md ...")

documents = []
files = glob.glob(os.path.join(DATA_FOLDER_PATH, "*.md"))

if not files:
    print("Tidak ada file .md ditemukan di folder tersebut!")
else:
    for file_path in files:
        try:
            filename = os.path.basename(file_path)
            print(f"🔄 Processing: {filename}...", end=" ", flush=True)

            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()

            text_input = extract_section(text, ["Nutrition Care Process",
                                    "Nutritional Care Process",
                                    "NUTRITIONAL CARE PROCESS",
                                    'Form Nutritional Care Process',
                                    'II. Form Nutritional Care Process',
                                    'BAB II. NCP',
                                    'FORM NUTRITIONAL CARE PROSES'
                                    ],
                                    ["BAB 3",
                                    "BAB III",
                                    'Perhitungan Kebutuhan',
                                    'RENCANA INTERVENSI'
                                    ], length=5000)

            response = client.responses.parse(
              model="gpt-4o-mini-2024-07-18",
              input=[
                  {"role": "system", "content": "Ekstrak informasi medis dari teks panjang yang diberikan"},
                  {"role": "user", "content": text_input},
              ],
              text_format=DataMedis,
            )

            extracted_data = response.output_parsed

            if extracted_data:
                meta_anthro = extracted_data.anthropometric_string
                meta_diag = extracted_data.medical_diagnosis
            else:
                meta_anthro = "Tidak Diketahui"
                meta_diag = "Tidak Diketahui"

            # Create LangChain Document
            doc = Document(
                page_content=text,
                metadata={
                    "source": filename,
                    "anthropometric_assessment": meta_anthro,
                    "medical_diagnosis": meta_diag
                }
            )
            documents.append(doc)
            # print("anthro", meta_anthro)
            # print('diag', meta_diag)
            print("✅ OK")

        except Exception as e:
            print(f"\n   ❌ Gagal load {filename}: {e}")

Sedang membaca file dari: /content/drive/Othercomputers/My Laptop/knowledge-base/processed-pymupdf/*.md ...
🔄 Processing: (HIL) DEXTRA INKASERATA + MECKEL’S DIVERTICUM POST EXPLORASI .md... ✅ OK
🔄 Processing: Abses Paru Post Evakuasi Abses + Wedge-repaired-ocr.md... ✅ OK
🔄 Processing: ANAK DENGAN TB PARU , PNEUMONIA , GIZI BURUK MARASMUS.md... ✅ OK
🔄 Processing: ANEMIA + SUSP LANGERHANS CELL HISTIOCYTOSIS + SUSP TU OTAK.md... ✅ OK
🔄 Processing: ARTHRALGIA + ANEMIA + HIPONATREMIA.md... ✅ OK
🔄 Processing: ASMA BRONKIAL.md... ✅ OK
🔄 Processing: ASTHMA SERANGAN BERAT, PNEUMONIA, INTRACTABLE EPILEPSI.md... ✅ OK
🔄 Processing: BAYI DENGAN NEONATUS ATERM + BERAT BADAN LAHIR RENDAH .md... ✅ OK
🔄 Processing: BAYI PREMATUR BBLR SMK (SESUAI MASA KEHAMILAN) LAHIR SC 3334 MINGGU.md... ✅ OK
🔄 Processing: BBLR, OEIS COMPLEX (OMPHALOCELE + EXTROFIA CLOACA + IMPERFRATA ANUS.md... ✅ OK
🔄 Processing: BERAT BADAN LAHIR SANGAT RENDAH (BBLSR), SESUAI MASA KEHAMILAN (SMK).md... ✅ OK
🔄 Processing: BERAT BADAN 

In [9]:
print(f"\nSedang memecah {len(documents)} dokumen...")

text_splitter = MarkdownTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

chunked_documents = text_splitter.split_documents(documents)
print(f"Total menjadi {len(chunked_documents)} chunks.")


Sedang memecah 120 dokumen...
Total menjadi 7243 chunks.


In [10]:
print(f"\nMenyimpan backup chunks ke {PICKLE_BACKUP_PATH}...")

with open(PICKLE_BACKUP_PATH, "wb") as f:
    pickle.dump(chunked_documents, f)


Menyimpan backup chunks ke backup_chunks_medis_v1.pkl...


In [11]:
print("\nMemulai proses Embedding & Upload ke Qdrant...")

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_ID,
    model_kwargs={
        "device": "cuda",
        "trust_remote_code": True,
        'token': HF_TOKEN
    }
)

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

if not client.collection_exists(COLLECTION_NAME):
    print(f"   - Membuat collection baru: {COLLECTION_NAME}")
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=256,
            distance=models.Distance.COSINE
        )
    )


Memulai proses Embedding & Upload ke Qdrant...


modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

   - Membuat collection baru: medical_reports_v1


In [12]:
vector_store = QdrantVectorStore.from_documents(
    documents=chunked_documents,
    embedding=embedding_model,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name=COLLECTION_NAME,
    force_recreate=True
)

print(f"\nSELESAI! {len(chunked_documents)} chunks berhasil masuk ke Qdrant.")


SELESAI! 7243 chunks berhasil masuk ke Qdrant.


## Synthetic Data Generation

In [13]:
data_generation_model = ChatOpenAI(model='gpt-4o-mini')
critic_model = ChatOpenAI(model='gpt-4o')

In [18]:
generator = TestsetGenerator.from_langchain(
    data_generation_model,
    critic_model,
    embedding_model
)

distributions = {
    simple: 0.3,         # 30% - Baseline test
    multi_context: 0.5,  # 50% - Fokus utama RRF
    reasoning: 0.2       # 20% - Uji penalaran LLM dari hasil RRF
}

In [19]:
custom_run_config = RunConfig(
    max_workers=1,     # Sangat Penting: Hanya proses 1 request dalam satu waktu (tidak paralel)
    max_retries=10,    # Jika kena rate limit, coba lagi hingga 10 kali
    max_wait=60        # Waktu tunggu maksimal antar retry (dalam detik)
)

In [21]:
import random

# Jika Anda butuh 30 soal, berikan sekitar 150 - 200 chunk saja sebagai bahan baku.
max_docs_for_ragas = 200

print(f"Total dokumen asli: {len(chunked_documents)}")

if len(chunked_documents) > max_docs_for_ragas:
    # Ambil sampel acak agar soal yang dihasilkan tetap bervariasi
    sampled_documents = random.sample(chunked_documents, max_docs_for_ragas)
    print(f"Menggunakan sampel: {len(sampled_documents)} dokumen.")
else:
    sampled_documents = chunked_documents

# 2. Masukkan dokumen yang SUDAH DISAMPEL ke dalam generator
print("Memulai generasi dataset...")
testset = generator.generate_with_langchain_docs(
    documents=sampled_documents,
    test_size=30,
    distributions=distributions,
    run_config=custom_run_config
)

Total dokumen asli: 7243
Menggunakan sampel: 200 dokumen.
Memulai generasi dataset...


embedding nodes:   0%|          | 0/428 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-lAr1djXZ6656SpL69sIqRRA1 on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}

In [ ]:
df_testset = testset.to_pandas()
df_testset.to_csv("dataset_evaluasi.csv", index=False)